# Transfer

> Train, cache, apply and adapt spectral models across datasets and instruments

In [ ]:
#| default_exp transfer

In [ ]:
#| export
import copy
from fastcore.all import *
import numpy as np, pandas as pd, torch
from fastai.torch_core import set_seed
from fastai.callback.schedule import *
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from soilspecdl.models import SpectralCNN
from soilspecdl.dataloader import TensorSpectrum, get_spectral_dls, get_splitter
from soilspecdl.transforms import BatchSNV
from soilspecdl.learner import spectral_learner

/Users/franckalbinet/aai-ws/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [ ]:
from fastcore.test import *
import tempfile
from sklearn.pipeline import make_pipeline
from soilspecdl.datasets import load_toy_kssl
from soilspecdl.models import CutCNN
from soilspecdl.dataloader import get_mask_splitter

A transfer experiment trains a model on a large spectral library, such as OSSL, then uses it on spectra from another dataset or instrument. It can apply the model as it is, recompute its normalisation statistics on the new spectra, fit a new head on its frozen features, or fine-tune it. The functions here do these steps for any `SpectralCNN` or `CutCNN`.

The examples use the toy dataset from `soilspecdl.datasets`: 500 KSSL MIR spectra. The target is `log1p` of exchangeable potassium (Kex), and every fifth sample is a test sample:

In [ ]:
toy = load_toy_kssl()
X, y, ids = toy.xy('k.ext_usda.a725_cmolc.kg')
y, wns = np.log1p(y.flatten()), toy.wavenumbers
is_test = np.arange(len(X)) % 5 == 0
X.shape, is_test.sum()

((500, 1701), np.int64(100))

## Training once

In [ ]:
#| export
def train_or_load(
    path, # File for the weights, such as `models/kex.pth`; the training history goes beside it, as a `.csv` file
    X, # Spectra, one per row
    y, # Targets
    wns, # Wavenumbers of the columns of `X`
    model_fn=SpectralCNN, # Function returning a new model
    epochs=20, # Number of one-cycle epochs
    lr=1e-3, # Maximum learning rate
    valid_pct=0.1, # Proportion of the samples for validation
    seed=42, # Seed for the validation split and the training
    tfm=BatchSNV(), # Transform applied to each batch
)->torch.nn.Module: # The trained model, on the CPU
    "Train a model from `model_fn` and save its weights to `path`, or load the weights if `path` exists"
    path = Path(path)
    if not path.exists():
        set_seed(seed)
        dls, _ = get_spectral_dls(X, y, wns, splitter=get_splitter(1 - valid_pct, valid_pct, seed=seed), batch_tfms=[tfm])
        learn = spectral_learner(dls, model_fn())
        with learn.no_bar(), learn.no_logging(): learn.fit_one_cycle(epochs, lr_max=lr)
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(learn.model.state_dict(), path)
        hist = pd.DataFrame(learn.recorder.values, columns=learn.recorder.metric_names[1:-1])
        hist.to_csv(path.with_suffix('.csv'), index_label='epoch')
    model = model_fn()
    model.load_state_dict(torch.load(path, map_location='cpu'))
    return model

Pretraining on a large library takes hours, and an experiment reuses the same pretrained model many times. `train_or_load` trains the model once, with a one-cycle schedule, and saves its weights to `path`. Later calls with the same `path` load the weights and skip the training. The function does not check that the arguments match the saved run. A new `path` is needed for every new setting.

Training also saves the losses and the validation R² of every epoch, in a `.csv` file beside the weights:

In [ ]:
path = Path(tempfile.mkdtemp())/'kex.pth'
model = train_or_load(path, X[~is_test], y[~is_test], wns, epochs=5)
hist = pd.read_csv(path.with_suffix('.csv'), index_col='epoch')
test_eq(len(hist), 5)
hist

,train_loss,valid_loss,r2_score
epoch,,,
0,0.157453,0.101628,-0.044382
1,0.143392,0.081949,0.157855
2,0.132672,0.092736,0.046997
3,0.129656,0.080547,0.172256
4,0.126212,0.073516,0.244508


## Predicting

In [ ]:
#| export
def _batched(
    f, # Function of the model and a batch, such as `lambda m, x: m(x)`
    model, # Trained model
    X, # Spectra, one per row
    wns, # Wavenumbers of the columns of `X`
    bs, # Batch size
    tfm, # Transform applied to each batch
)->np.ndarray: # Outputs of `f`, concatenated over the batches
    "Apply `f` to `X` in batches, after `tfm`, with a copy of `model` in evaluation mode on the CPU"
    m, out = copy.deepcopy(model).cpu().eval(), []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            xb = TensorSpectrum(torch.tensor(X[i:i+bs], dtype=torch.float32).unsqueeze(1), wns=wns)
            out.append(f(m, tfm(xb)).numpy())
    return np.concatenate(out)

def predict(
    model, # Trained model
    X, # Spectra, one per row
    wns, # Wavenumbers of the columns of `X`
    bs=256, # Batch size
    tfm=BatchSNV(), # Transform applied to each batch, as in training
)->np.ndarray: # One prediction per spectrum
    "Predict `X` with `model`"
    return _batched(lambda m, x: m(x), model, X, wns, bs, tfm)

`predict` needs no `DataLoaders`. It applies the same batch transform as training, `BatchSNV` by default, and works on a copy of the model in evaluation mode. A second `train_or_load` with the same `path` loads the saved weights, and gives the same predictions:

In [ ]:
p = predict(model, X[is_test], wns)
test_close(predict(train_or_load(path, X[~is_test], y[~is_test], wns), X[is_test], wns), p)
p.shape

(100,)

## Scores

In [ ]:
#| export
def regression_scores(
    y, # Observed targets
    p, # Predicted targets
    inv=None, # Inverse of the target transform, such as `np.expm1`, for RMSE and bias on the property's own scale
)->dict: # R², correlation `r` and concordance `CCC` of the targets; RMSE and bias of the property
    "R², correlation, Lin's concordance correlation coefficient, RMSE and bias of predictions"
    err = inv(p) - inv(y) if inv else p - y
    ccc = 2 * np.cov(y, p, bias=True)[0, 1] / (np.var(y) + np.var(p) + (np.mean(y) - np.mean(p))**2)
    res = {'R²': r2_score(y, p), 'r': np.corrcoef(y, p)[0, 1], 'CCC': ccc, 'RMSE': np.sqrt(np.mean(err**2)), 'bias': np.mean(err)}
    return {k: float(v) for k, v in res.items()}

Lin's concordance correlation coefficient (CCC) measures agreement with the 1:1 line. Unlike `r`, it drops when predictions are biased or on the wrong scale. Safanelli et al. (2023) used it as their main metric for the soil spectroscopy ring trial. Models that predict a transformed target, such as `log1p` of a property, are often judged on the property itself. With `inv`, RMSE and bias are computed after the inverse transform, in the property's own units. R², `r` and CCC stay on the transformed scale:

In [ ]:
scores = regression_scores(y[is_test], p, inv=np.expm1)
test_close(regression_scores(y, y)['CCC'], 1)
scores

{'R²': 0.22418335926025057,
 'r': 0.6029749923404765,
 'CCC': 0.4288575795203078,
 'RMSE': 0.5868559092121325,
 'bias': 0.10391348495597841}

## Frozen features

In [ ]:
#| export
def features(
    model, # Model with a `backbone`, such as a `SpectralCNN` or a `CutCNN`
    X, # Spectra, one per row
    wns, # Wavenumbers of the columns of `X`
    bs=256, # Batch size
    tfm=BatchSNV(), # Transform applied to each batch, as in training
)->np.ndarray: # One row of features per spectrum
    "The outputs of the backbone of `model` for the spectra `X`, which are the inputs of its head"
    return _batched(lambda m, x: m.backbone(x), model, X, wns, bs, tfm)

The backbone of a `SpectralCNN` gives one value per channel of its last block. A `CutCNN` cut after four blocks gives the 128 channels of block 4:

In [ ]:
cut = CutCNN(model, len(wns))
test_eq(features(model, X[:10], wns).shape, (10, 256))
features(cut, X[:10], wns).shape

(10, 128)

In [ ]:
#| export
def ridge_head(
    model, # Model with a `backbone` and a `head` ending in a `Linear` layer
    X, # Training spectra, one per row
    y, # Training targets
    wns, # Wavenumbers of the columns of `X`
    alphas=np.logspace(-1, 6, 15), # Ridge penalties to choose from by leave-one-out cross-validation
    tfm=BatchSNV(), # Transform applied to each batch, as in training
)->torch.nn.Module: # Copy of `model` with the ridge regression in its head
    "Copy of `model` whose head holds a ridge regression of `y` on the features of `X`"
    m, F = copy.deepcopy(model), features(model, X, wns, tfm=tfm)
    sc = StandardScaler().fit(F)
    r = RidgeCV(alphas=alphas).fit(sc.transform(F), y)
    w = r.coef_ / sc.scale_
    with torch.no_grad():
        m.head[-1].weight.copy_(torch.tensor(w[None], dtype=torch.float32))
        m.head[-1].bias.fill_(float(r.intercept_ - (w * sc.mean_).sum()))
    return m

With few target samples, a ridge regression on frozen features can beat fine-tuning. `ridge_head` standardises the features, fits `RidgeCV`, and folds the scaling into the weights of the head's last `Linear` layer. The result is an ordinary model: `predict`, `spectral_learner` and fine-tuning work on it. Its predictions match a scikit-learn pipeline fitted on the same features:

In [ ]:
rh = ridge_head(cut, X[~is_test], y[~is_test], wns)
F = features(cut, X[~is_test], wns)
pipe = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-1, 6, 15))).fit(F, y[~is_test])
test_close(predict(rh, X[~is_test], wns), pipe.predict(F), eps=1e-4)
regression_scores(y[is_test], predict(rh, X[is_test], wns), inv=np.expm1)

{'R²': 0.415871848976444,
 'r': 0.6712359540465389,
 'CCC': 0.5915279429029128,
 'RMSE': 0.5306872989165431,
 'bias': 0.027256447220993064}

## Adapting to new spectra

In [ ]:
#| export
def adabn(
    model, # Trained model
    dls, # `DataLoaders` of the new spectra
)->torch.nn.Module: # Copy of `model`, in evaluation mode
    "Copy of `model` whose BatchNorm running statistics are recomputed from the training and validation spectra of `dls`"
    m = copy.deepcopy(model).to(dls.device).eval()
    bns = [l for l in m.modules() if isinstance(l, torch.nn.modules.batchnorm._BatchNorm)]
    for bn in bns: bn.reset_running_stats(); bn.momentum = None; bn.train()
    with torch.no_grad():
        for dl in (dls.train.new(shuffle=False, drop_last=False), dls.valid):
            for xb, _ in dl: m(xb)
    return m.eval()

def test_preds(
    learn, # `Learner` whose `dls` comes from `get_spectral_dls`
)->np.ndarray: # One prediction per test sample
    "Predict the test samples `learn.dls.test_idx`"
    with learn.no_bar(): p, _ = learn.get_preds(dl=learn.dls.test_dl(learn.dls.test_idx.tolist(), with_labels=True))
    return p.numpy().ravel()

Spectra from another instrument shift the activations inside the network. BatchNorm layers normalise with running means and variances learned during training, which then no longer fit, and each layer receives off-centre inputs. AdaBN (Li et al., 2016) resets the statistics and recomputes them from the new spectra. `adabn` runs the training and validation spectra of `dls` through a copy of the model, with BatchNorm in training mode and `momentum=None`. Each running statistic becomes the average of the per-batch statistics, with every batch weighted equally. `adabn` changes no weights, uses no targets, and leaves the test spectra out. Afterwards, the running mean of the first BatchNorm layer is the average over batches of the mean output of the first convolution:


In [ ]:
dls, _ = get_spectral_dls(X, y, wns, splitter=get_mask_splitter(is_test), batch_tfms=[BatchSNV()])
ada = adabn(model, dls)
conv, bn = ada.backbone.layers[0].conv[0], ada.backbone.layers[0].conv[1]
with torch.no_grad(): outs = [conv(xb) for dl in (dls.train.new(shuffle=False, drop_last=False), dls.valid) for xb, _ in dl]
test_close(bn.running_mean.cpu().numpy(), torch.stack([o.mean(dim=(0, 2)) for o in outs]).mean(0).cpu().numpy(), eps=1e-6)


{'R²': 0.33156271998704046,
 'r': 0.5971741783023902,
 'CCC': 0.47963940503303787,
 'RMSE': 0.5793957682479973,
 'bias': -0.00779515136514901}

`test_preds` predicts the test samples of the `DataLoaders` that a `Learner` uses. Here the new spectra come from the same instrument as the training spectra, so AdaBN changes the predictions only a little:

In [ ]:
p_ada = test_preds(spectral_learner(dls, ada))
regression_scores(y[is_test], p_ada, inv=np.expm1)